# Raw Data Exploration & Preprocessing Audit

Exploratory notebook for inspecting raw datasets (`01_raw`) and prototyping stateless cleaning functions before refactoring them into `src/album_recs/pipelines/data_processing/nodes.py` (`02_intermediate`).

* **`playlists_dump.sql` (Playlist2vec Relational Dump):** Stream-sampled below (50,000 rows per table) across `playlist`, `track`, `album`, and `track_playlist1` to diagnose raw anomalies and prototype table-level cleaning functions.

In [3]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [ ]:
# this scans the file in a single pass so it picks up every CREATE TABLE and samples 50,000 rows from all tables
import re
import pandas as pd

# Text patterns loaded once up front to safely read the SQL file line by line:
# - TOKEN_RE: splits data rows into '(', ')', 'quoted text' (keeping commas/apostrophes inside titles intact), and unquoted values (NULL or numbers)
# - CREATE_RE / COL_RE: grabs table and column names from table setup blocks while skipping index/key rules
# - INSERT_RE: finds INSERT INTO data lines (whether table names use quotes or not) and marks where the row values begin
TOKEN_RE = re.compile(r"\(|\)|'((?:\\.|''|[^'\\])*)'|([^,()\s';]+)")
CREATE_RE = re.compile(r"^\s*CREATE\s+TABLE(?:\s+IF\s+NOT\s+EXISTS)?\s+[`\"]?(\w+)[`\"]?", re.I)
INSERT_RE = re.compile(r"^\s*INSERT(?:\s+IGNORE)?\s+INTO\s+[`\"]?(\w+)[`\"]?(?:\s*\(([^)]+)\))?\s+VALUES\s*", re.I)
COL_RE = re.compile(r"^\s*[`\"]?(\w+)[`\"]?\s+(?:varchar|char|text|int|bigint|smallint|tinyint|mediumint|float|double|decimal|date|datetime|timestamp|bool|boolean|blob|enum)\b", re.I)

# Reads a SQL VALUES line piece by piece and yields one clean Python list per row:
# - Starts a new row list at '(' and outputs the finished row at ')'
# - Unescapes SQL characters inside 'quoted text' (like \' or \n) back into normal text
# - Turns unquoted NULL values into Python None (which Pandas treats as missing/NaN)
def _iter_mysql_rows(values_str: str):
    row = None
    for m in TOKEN_RE.finditer(values_str):
        tok = m.group(0)
        if tok == "(":
            row = []
        elif tok == ")":
            if row is not None:
                yield row
                row = None
        elif row is not None:
            if m.group(1) is not None:
                val = (
                    m.group(1)
                    .replace("''", "'")
                    .replace("\\'", "'")
                    .replace('\\"', '"')
                    .replace("\\n", "\n")
                    .replace("\\r", "\r")
                    .replace("\\t", "\t")
                    .replace("\\\\", "\\")
                )
                row.append(val)
            else:
                raw = m.group(2)
                row.append(None if raw == "NULL" else raw)

In [ ]:
# Streams the SQL file line by line (without loading all 10.7 GB into RAM) and returns a dict of DataFrames:
# - Quickly skips blank/comment lines until it sees a CREATE TABLE or INSERT INTO statement
# - Saves each table's column names from CREATE TABLE blocks (or inline INSERT column lists)
# - Unpacks data rows from INSERT INTO lines until a table hits max_rows_per_table (e.g., 50,000), then skips the rest
# - Turns each table's collected rows into a Pandas DataFrame (using col_0, col_1... if column headers were missing)
def parse_sql_dump(sql_path, max_rows_per_table: int | None = None) -> dict[str, pd.DataFrame]:
    schemas: dict[str, list[str]] = {}
    rows: dict[str, list] = {}
    current_table = None

    with open(sql_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            # skip lines that cannot be CREATE, column defs, or INSERT
            if current_table is None and "INSERT" not in line and "CREATE" not in line:
                continue

            # 1. Detect start of a CREATE TABLE block
            if "CREATE" in line:
                m_create = CREATE_RE.match(line)
                if m_create:
                    current_table = m_create.group(1)
                    schemas.setdefault(current_table, [])
                    rows.setdefault(current_table, [])
                    continue

            # Inside a CREATE TABLE block: grab column names until the closing ')'
            if current_table is not None:
                if re.match(r"^\s*\)", line):
                    current_table = None
                else:
                    m_col = COL_RE.match(line)
                    if m_col:
                        schemas[current_table].append(m_col.group(1))
                continue

            # 2. Capture INSERT INTO rows for ANY table (even if no CREATE TABLE was seen)
            if "INSERT" in line:
                m_ins = INSERT_RE.match(line)
                if not m_ins:
                    continue

                table = m_ins.group(1)
                rows.setdefault(table, [])

                # Stop collecting once this table hits the sample limit
                if max_rows_per_table and len(rows[table]) >= max_rows_per_table:
                    continue

                # Grab inline column names if formatted as INSERT INTO tbl (col1, col2) VALUES ...
                if m_ins.group(2) and table not in schemas:
                    schemas[table] = [c.strip(" `\"") for c in m_ins.group(2).split(",")]

                values_part = line[m_ins.end():]
                for row in _iter_mysql_rows(values_part):
                    rows[table].append(row)
                    if max_rows_per_table and len(rows[table]) >= max_rows_per_table:
                        break

    # 3. Convert each table's rows into a DataFrame (falling back to col_0, col_1... if headers were missing)
    tables: dict[str, pd.DataFrame] = {}
    for table, table_rows in rows.items():
        if not table_rows:
            continue
        cols = schemas.get(table)
        if not cols or len(cols) != len(table_rows[0]):
            cols = [f"col_{i}" for i in range(len(table_rows[0]))]
        tables[table] = pd.DataFrame(table_rows, columns=cols)

    return tables

In [ ]:
# Get the SQL file path from catalog.yml (without loading all 10.7 GB into RAM)
dump_path = catalog.load("playlists_dump")

# Load a 50k-row sample per table in seconds to inspect columns and dtypes
sample_tables = parse_sql_dump(dump_path, max_rows_per_table=50_000)

[11/16/26 16:02:09] INFO     Loading data from playlists_dump (RawFilePathDataset)...          ]8;id=14429321;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=14429322;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\io\data_catalog.py#1238\1238]8;;\

In [7]:
# Pull out only the 4 relational tables needed for your album recommender
# Don't need artist or track_artist1 as processed_tracks has primary_artist and recommender is on album level
tables_to_explore = {
    "playlist": sample_tables["playlist"],
    "track": sample_tables["track"],
    "album": sample_tables["album"],
    "track_playlist1": sample_tables["track_playlist1"],
}

In [8]:
for name, df in tables_to_explore.items():
    print(f"\n{'='*40}\nTABLE: {name} | Shape: {df.shape}\n{'='*40}")
    
    # 1. Column names & types -> tells you what to rename and cast to Int64
    print("Columns & dtypes:\n", df.dtypes)
    
    # 2. True nulls (NaN) -> tells you which foreign keys have missing values
    print("\nNull (NaN) counts:\n", df.isnull().sum()[df.isnull().sum() > 0])
    
    # 3. Empty strings ("") -> catches local MP3 rows that slip past .dropna()
    empty_counts = (df == "").sum()
    print("\nEmpty string ('') counts:\n", empty_counts[empty_counts > 0])
    
    # 4. Duplicate check on the first column (primary key)
    pk = df.columns[0]
    print(f"\nDuplicates on `{pk}`:", df[pk].duplicated().sum())
    
    display(df.head(3))


TABLE: playlist | Shape: (50000, 5)
Columns & dtypes:
 id              str
name            str
followers       str
uri             str
total_tracks    str
dtype: object

Null (NaN) counts:
 followers    57
dtype: int64

Empty string ('') counts:
 Series([], dtype: int64)

Duplicates on `id`: 0


,id,name,followers,uri,total_tracks
0,0001o8bRlvQld8j74gpOuo,chillwave,0,spotify:user:1156860855:playlist:0001o8bRlvQld...,147
1,0003EwMBT8iPys9zW1KWvu,Vaporwave Race 64,0,spotify:user:1261134915:playlist:0003EwMBT8iPy...,30
2,0005GNkME45OHjWo7sW08R,baile funk,1,spotify:user:giovannapaz.santos:playlist:0005G...,115



TABLE: track | Shape: (50000, 8)
Columns & dtypes:
 id             str
name           str
duration       str
popularity     str
explicit       str
preview_url    str
uri            str
album_id       str
dtype: object

Null (NaN) counts:
 preview_url    15012
album_id           1
dtype: int64

Empty string ('') counts:
 id       1
name    23
dtype: int64

Duplicates on `id`: 0


,id,name,duration,popularity,explicit,preview_url,uri,album_id
0,,The Grudge,516000,0,0,NaN,spotify:local:Tool::The+Grudge:516,NaN
1,000075Sys6SZvjmUtYspEr,Profetizando el Apocalipsis,255240,0,0,https://p.scdn.co/mp3-preview/2df326ba0c688ebe...,spotify:track:000075Sys6SZvjmUtYspEr,4quYGjnXnL4xuIpSxNFrGE
2,0000iGYsxbrtG0PNnL2Mw2,Forbudte følelser,105600,11,0,NaN,spotify:track:0000iGYsxbrtG0PNnL2Mw2,4NPe1Iq9dS7vnFBbyc83Um



TABLE: album | Shape: (50000, 3)
Columns & dtypes:
 id      str
name    str
uri     str
dtype: object

Null (NaN) counts:
 uri    1
dtype: int64

Empty string ('') counts:
 id       1
name    16
dtype: int64

Duplicates on `id`: 0


,id,name,uri
0,,,NaN
1,0000GXfwK4NuQ91j70P5Ex,Devocean,spotify:album:0000GXfwK4NuQ91j70P5Ex
2,0000LFuOuVDo60CnTLknzP,The Walker,spotify:album:0000LFuOuVDo60CnTLknzP



TABLE: track_playlist1 | Shape: (50000, 2)
Columns & dtypes:
 track_id       str
playlist_id    str
dtype: object

Null (NaN) counts:
 Series([], dtype: int64)

Empty string ('') counts:
 Series([], dtype: int64)

Duplicates on `track_id`: 44149


,track_id,playlist_id
0,000075Sys6SZvjmUtYspEr,6qXdp3KgFD1gamKd9xZqvP
1,0000iGYsxbrtG0PNnL2Mw2,1BzaYDosDh5ZVRRX5mzuyb
2,0000iGYsxbrtG0PNnL2Mw2,50kEMRMW3vZ2LwIy55F0pP


### Cleaning Decisions Based on Diagnostic Findings
From the 50k-row sample above, we define four table-specific preprocessing functions (plus a shared `_parse_nullable_int` helper) to fix five issues before moving the logic into `nodes.py`:

1. Rename generic `id` and `name` columns.
2. Convert empty strings to `pd.NA` before `.dropna()`.
3. Cast numeric strings to nullable `Int64`.
4. Drop `uri` and `preview_url`.
5. Deduplicate `track_playlist1` on `["track_id", "playlist_id"]` (44,149 repeated `track_id` values confirm this is a many-to-many junction table, so deduplicating on `track_id` alone would delete valid playlist appearances).

In [ ]:
# Converts numeric text to whole numbers while safely keeping missing values (Int64)
def _parse_nullable_int(x: pd.Series) -> pd.Series:
    return pd.to_numeric(x, errors="coerce").astype("Int64")

In [10]:
def preprocess_playlists(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(
        columns={
            "id": "playlist_id",
            "name": "playlist_name",
        }
    )
    df = df.replace("", pd.NA)
    df = df.dropna(subset=["playlist_id"]).drop_duplicates(subset=["playlist_id"])
    df = df.drop(columns=["uri"], errors="ignore")

    df["followers"] = _parse_nullable_int(df["followers"])
    df["total_tracks"] = _parse_nullable_int(df["total_tracks"])
    return df


def preprocess_sql_tracks(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(
        columns={
            "id": "track_id",
            "name": "track_name",
            "duration": "duration_ms",
        }
    )
    df = df.replace("", pd.NA)
    df = df.dropna(
        subset=["track_id", "album_id", "track_name"]
    ).drop_duplicates(subset=["track_id"])
    df = df.drop(columns=["uri", "preview_url"], errors="ignore")

    df["duration_ms"] = _parse_nullable_int(df["duration_ms"])
    df["popularity"] = _parse_nullable_int(df["popularity"])
    df["explicit"] = _parse_nullable_int(df["explicit"])
    return df


def preprocess_sql_albums(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(
        columns={
            "id": "album_id",
            "name": "album_name",
        }
    )
    df = df.replace("", pd.NA)
    df = df.dropna(subset=["album_id", "album_name"]).drop_duplicates(
        subset=["album_id"]
    )
    df = df.drop(columns=["uri"], errors="ignore")
    return df


def preprocess_playlist_tracks(df: pd.DataFrame) -> pd.DataFrame:
    df = df.replace("", pd.NA)
    df = df.dropna(subset=["track_id", "playlist_id"]).drop_duplicates(
        subset=["track_id", "playlist_id"]
    )
    return df

In [ ]:
playlists_clean = preprocess_playlists(sample_tables["playlist"])
sql_tracks_clean = preprocess_sql_tracks(sample_tables["track"])
sql_albums_clean = preprocess_sql_albums(sample_tables["album"])
playlist_tracks_clean = preprocess_playlist_tracks(sample_tables["track_playlist1"])

# Verify each cleaned table's shape, column types (e.g., Int64), and first 2 rows
for name, df in {
    "playlists_clean": playlists_clean,
    "sql_tracks_clean": sql_tracks_clean,
    "sql_albums_clean": sql_albums_clean,
    "playlist_tracks_clean": playlist_tracks_clean,
}.items():
    print(f"\n=== {name} | Shape: {df.shape} ===")
    print(df.dtypes)
    display(df.head(2))


=== playlists_clean | Shape: (50000, 4) ===
playlist_id        str
playlist_name      str
followers        Int64
total_tracks     Int64
dtype: object


,playlist_id,playlist_name,followers,total_tracks
0,0001o8bRlvQld8j74gpOuo,chillwave,0,147
1,0003EwMBT8iPys9zW1KWvu,Vaporwave Race 64,0,30



=== sql_tracks_clean | Shape: (49976, 6) ===
track_id         str
track_name       str
duration_ms    Int64
popularity     Int64
explicit       Int64
album_id         str
dtype: object


,track_id,track_name,duration_ms,popularity,explicit,album_id
1,000075Sys6SZvjmUtYspEr,Profetizando el Apocalipsis,255240,0,0,4quYGjnXnL4xuIpSxNFrGE
2,0000iGYsxbrtG0PNnL2Mw2,Forbudte følelser,105600,11,0,4NPe1Iq9dS7vnFBbyc83Um



=== sql_albums_clean | Shape: (49984, 2) ===
album_id      str
album_name    str
dtype: object


,album_id,album_name
1,0000GXfwK4NuQ91j70P5Ex,Devocean
2,0000LFuOuVDo60CnTLknzP,The Walker



=== playlist_tracks_clean | Shape: (50000, 2) ===
track_id       str
playlist_id    str
dtype: object


,track_id,playlist_id
0,000075Sys6SZvjmUtYspEr,6qXdp3KgFD1gamKd9xZqvP
1,0000iGYsxbrtG0PNnL2Mw2,1BzaYDosDh5ZVRRX5mzuyb


## Post-Pipeline Audit (`02_intermediate`)

Loads all five preprocessed Parquet datasets produced by the `data_processing` pipeline via `catalog.load(...)` to validate the full ingestion run.

In [ ]:
%reload_ext kedro.ipython
import pandas as pd

datasets = {
    "tracks (Kaggle)": catalog.load("preprocessed_tracks"),
    "playlists (SQL)": catalog.load("preprocessed_playlists"),
    "sql_tracks (SQL)": catalog.load("preprocessed_sql_tracks"),
    "sql_albums (SQL)": catalog.load("preprocessed_sql_albums"),
    "playlist_tracks (SQL)": catalog.load("preprocessed_playlist_tracks"),
}

pk_map = {
    "tracks (Kaggle)": ["track_id"],
    "playlists (SQL)": ["playlist_id"],
    "sql_tracks (SQL)": ["track_id"],
    "sql_albums (SQL)": ["album_id"],
    "playlist_tracks (SQL)": ["track_id", "playlist_id"],
}

for name, df in datasets.items():
    print(f"\n{'='*50}\n{name} | Shape: {df.shape[0]:,} rows × {df.shape[1]} cols\n{'='*50}")
    print("Dtypes:\n", df.dtypes)
    
    # Check for unexpected nulls
    nulls = df.isnull().sum()
    print("\nNull counts:\n", nulls[nulls > 0] if (nulls > 0).any() else "None")
    
    # Confirm zero empty strings ("") survived
    str_cols = df.select_dtypes(include=["object", "string"]).columns
    empty_strs = {c: (df[c] == "").sum() for c in str_cols if (df[c] == "").sum() > 0}
    print("Empty strings (''):", empty_strs if empty_strs else "None")
    
    # Confirm primary keys are 100% unique across the entire dataset
    dupes = df.duplicated(subset=pk_map[name]).sum()
    print(f"Duplicate keys on {pk_map[name]}: {dupes}")
    
    display(df.head(2))

[11/16/26 19:33:52] INFO     Registered line magic '%reload_kedro'                                   ]8;id=14429327;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=14429328;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\ipython\__init__.py#70\70]8;;\

                    INFO     Registered line magic '%load_node'                                      ]8;id=14429333;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=14429334;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\ipython\__init__.py#72\72]8;;\

                    INFO     Resolved project path as: c:\projects\album-recs\album-recs.           ]8;id=14429339;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=14429340;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\ipython\__init__.py#261\261]8;;\
                             To set a different path, run '%reload_kedro <project_root>'                           

[11/16/26 19:33:53] INFO     No typed parameter requirements found, returning original   ]8;id=14429345;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\validation\parameter_validator.py\parameter_validator.py]8;;\:]8;id=14429346;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\validation\parameter_validator.py#124\124]8;;\
                             parameters                                                                            

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=14429351;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=14429352;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro_telemetry\plugin.py#279\279]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project album-recs                                               ]8;id=14429357;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=14429358;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\ipython\__init__.py#227\227]8;;\

... (13 outputs left out)


sql_albums (SQL) | Shape: 2,917,411 rows × 2 cols
Dtypes:
 album_id      str
album_name    str
dtype: object

Null counts:
 None
Empty strings (''): None
Duplicate keys on ['album_id']: 0


,album_id,album_name
0,0000GXfwK4NuQ91j70P5Ex,Devocean
1,0000LFuOuVDo60CnTLknzP,The Walker



playlist_tracks (SQL) | Shape: 125,451,006 rows × 2 cols
Dtypes:
 track_id       str
playlist_id    str
dtype: object

Null counts:
 None
Empty strings (''): None
Duplicate keys on ['track_id', 'playlist_id']: 0


,track_id,playlist_id
0,000075Sys6SZvjmUtYspEr,6qXdp3KgFD1gamKd9xZqvP
1,0000iGYsxbrtG0PNnL2Mw2,1BzaYDosDh5ZVRRX5mzuyb


In [13]:
playlists = datasets["playlists (SQL)"]
sql_tracks = datasets["sql_tracks (SQL)"]
sql_albums = datasets["sql_albums (SQL)"]
pt = datasets["playlist_tracks (SQL)"]
kaggle_tracks = datasets["tracks (Kaggle)"]

# 1. Internal SQL consistency (should be ~99-100%)
valid_playlists = pt["playlist_id"].isin(set(playlists["playlist_id"])).mean() * 100
valid_sql_tracks = pt["track_id"].isin(set(sql_tracks["track_id"])).mean() * 100
valid_sql_albums = sql_tracks["album_id"].isin(set(sql_albums["album_id"])).mean() * 100

print(f"playlist_tracks matching a valid playlist_id: {valid_playlists:.2f}%")
print(f"playlist_tracks matching a valid SQL track_id: {valid_sql_tracks:.2f}%")
print(f"sql_tracks matching a valid SQL album_id:     {valid_sql_albums:.2f}%")

# 2. Cross-source overlap (SQL Playlist2vec <-> Kaggle 1.2M Audio Features)
shared_tracks = len(set(sql_tracks["track_id"]) & set(kaggle_tracks["track_id"]))
shared_albums = len(set(sql_tracks["album_id"]) & set(kaggle_tracks["album_id"]))

print(f"\nShared track_ids between SQL dump & Kaggle CSV: {shared_tracks:,}")
print(f"Shared album_ids between SQL dump & Kaggle CSV: {shared_albums:,}")

playlist_tracks matching a valid playlist_id: 100.00%
playlist_tracks matching a valid SQL track_id: 99.98%
sql_tracks matching a valid SQL album_id:     100.00%

Shared track_ids between SQL dump & Kaggle CSV: 389,815
Shared album_ids between SQL dump & Kaggle CSV: 49,467
